In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

In [2]:
# 1. DATA INGESTION & ROBUST PREPROCESSING
print("Step 1: Ingesting customer account ledger...")
df = pd.read_csv('/content/customer_churn_ann_dataset1_55d8b8a8-aa58-4c01-b459-d30aeda770d2_268917_.csv')

Step 1: Ingesting customer account ledger...


In [3]:
# Drop CustomerID as it is metadata and holds no behavioral value
df = df.drop(columns=['CustomerID'])

In [4]:
# DATA CLEANING: Fill missing values in InternetService with a fallback category
df['InternetService'] = df['InternetService'].fillna('Unknown')

In [6]:
# Isolate features and target label
X_raw = df.drop(columns=['Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1}) # Encode binary target to integers

In [7]:
# Transform categorical strings into numerical structural flags via One-Hot Encoding
X_encoded = pd.get_dummies(X_raw, columns=['Contract', 'InternetService','TechSupport', 'PaymentMethod', 'PaperlessBilling'], drop_first=True)

In [8]:
# Save the exact schema template to align incoming manual traffic during live inference
TRAINING_FEATURE_SCHEMA = X_encoded.columns.tolist()

In [9]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
# 2. FEATURE SCALING (CRITICAL FOR ANNs)
print("Step 2: Standardizing continuous feature matrices...")
# Neural networks calculate weights using gradient optimization. If input features
# have highly mismatched scales, the gradients will oscillate, slowing down covergence
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

Step 2: Standardizing continuous feature matrices...


In [12]:
# 3. TRAINING THE MULTI-LAYER PERCEPTRON (ANN)
print("Step 3: Stacking hidden layers and fitting the network parameters...")

Step 3: Stacking hidden layers and fitting the network parameters...


In [13]:
# hidden_layer_sizes=(16, 8) builds an architecture with 16 neurons in Hidden Layer 1,
# and 8 neurons in Hidden Layer 2. 'adam' is a robust stochastic gradient optimizer.
mlp_ann = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)
mlp_ann.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42)

In [14]:
# 4. SYSTEM PERFORMANCE AUDIT
print("\n=== COGNITIVE ANN QUALITY PERFORMANCE DASHBOARD ===")
y_pred = mlp_ann.predict(X_test)
print(f"Neural Network Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Retained Account', 'Churned Account']))


=== COGNITIVE ANN QUALITY PERFORMANCE DASHBOARD ===
Neural Network Accuracy: 92.50%
                  precision    recall  f1-score   support

Retained Account       0.97      0.94      0.95        31
 Churned Account       0.80      0.89      0.84         9

        accuracy                           0.93        40
       macro avg       0.88      0.91      0.90        40
    weighted avg       0.93      0.93      0.93        40



In [16]:
# 5. LIVE INFERENCE ENGINE (REALISTIC MANUAL ENTRY)
print("\n=== REAL-TIME INFERENCE SERVICE ENGINE (MANUAL DATA ENTRY)===")
# Enter raw data fields manually, exactly as they arrive from an external account CRM portal
unseen_raw_accounts = pd.DataFrame([
    {
        'Age': 24, 'TenureMonths': 2, 'MonthlyCharges': 105, 'TotalCharges': 210,
        'NumSupportTickets': 9, 'Contract': 'Month-to-month', 'InternetService': 'Fiber',
        'TechSupport': 'No', 'PaymentMethod': 'Electronic check', 'PaperlessBilling': 'Yes'
    },
    {
        'Age': 58, 'TenureMonths': 68, 'MonthlyCharges': 45, 'TotalCharges': 3060,
        'NumSupportTickets': 0, 'Contract': 'Two year', 'InternetService': 'DSL',
        'TechSupport': 'Yes', 'PaymentMethod': 'Bank transfer', 'PaperlessBilling': 'No'
    }
])


=== REAL-TIME INFERENCE SERVICE ENGINE (MANUAL DATA ENTRY)===


In [17]:
# Process manual fields using the exact same dummies template rule
unseen_encoded = pd.get_dummies(unseen_raw_accounts, columns=['Contract','InternetService', 'TechSupport', 'PaymentMethod', 'PaperlessBilling'])

In [19]:
# Align input columns to match our training dataset shape perfectly
unseen_aligned = unseen_encoded.reindex(columns=TRAINING_FEATURE_SCHEMA, fill_value=0)

In [23]:
# Scale using the existing baseline transformation mapping
unseen_scaled = scaler.transform(unseen_aligned)

In [25]:
# Extract binary classifications and raw probability values
live_verdicts = mlp_ann.predict(unseen_scaled)
live_probabilities = mlp_ann.predict_proba(unseen_scaled)[:, 1]

In [30]:
# Display structured results
inference_dashboard= pd.DataFrame({
    'Account Key':['Customer Alpha (High Support User)','Customer Beta (Long Term Acount)'],
    'Risk Probability': live_probabilities,
    'System Prediction Code': live_verdicts
})

In [33]:
inference_dashboard['Retention Retaining Plan'] = inference_dashboard['System Prediction Code'].map({
    0: 'Account is healthy. Maintain regular billing schedules.',
    1:'CRTICAL CHURN RISK! Dispatch automated retention incentive coupon immediately'
})
print(inference_dashboard[[
    'Account Key', 'Risk Probability', 'Retention Retaining Plan'
]].to_string(index=False))

                       Account Key  Risk Probability                                                      Retention Retaining Plan
Customer Alpha (High Support User)          0.999930 CRTICAL CHURN RISK! Dispatch automated retention incentive coupon immediately
  Customer Beta (Long Term Acount)          0.000003                       Account is healthy. Maintain regular billing schedules.
